# FinFlow Automated Verification — Challenge Pre/Post Analysis

This notebook is the independent practice version of the FinFlow Pre/Post case. It intentionally removes the worked answers. Your task is to evaluate whether the automated verification workflow improved transaction-processing performance while keeping guardrails acceptable.

**Intervention date:** 2026-04-01  
**Primary KPI:** verification completion rate  
**Secondary KPIs:** verification time, manual review rate, completed transaction volume  
**Guardrails:** payment decline rate, support contact rate, fraud-confirmed rate

All data are synthetic and generated for training and portfolio demonstration only.

## Challenge objective

Build a complete Pre/Post impact analysis from raw synthetic data to stakeholder recommendation. Your analysis should avoid the common mistake of treating any post-launch movement as causal proof.

Use the framework: business question → data quality → KPI movement → trend/seasonality → confounders → ITS → guardrails → business impact → recommendation.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
module_root = None
for candidate in [cwd] + list(cwd.parents):
    if (candidate / 'src' / 'generate_synthetic_data.py').exists() and (candidate / 'src' / 'analyze_pre_post.py').exists():
        module_root = candidate
        break
    if (candidate / '02_pre_post_analysis' / 'src' / 'generate_synthetic_data.py').exists():
        module_root = candidate / '02_pre_post_analysis'
        break

if module_root is None:
    raise FileNotFoundError('Could not locate the 02_pre_post_analysis module root.')

sys.path.insert(0, str(module_root / 'src'))

from generate_synthetic_data import generate_raw_data
from analyze_pre_post import (
    LAUNCH_DATE, RAMP_END, CAMPAIGN_START, CAMPAIGN_END,
    quality_report, clean_data, kpi_summary, two_proportion_pre_post,
    continuous_pre_post, daily_aggregate, interrupted_time_series,
    mix_comparison, segment_completion, stable_post_mask
)

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Business framing

Write 3–5 sentences answering:
- What decision is FinFlow trying to make?
- What would a successful launch look like?
- Why is this not equivalent to an A/B test?
- What are the main causal risks in this design?

## 2. Generate and inspect raw data

Generate the raw synthetic data, inspect the first rows, and produce the data-quality report. Identify the issues that must be corrected before analysis.

In [ ]:
# TODO: Generate raw data and inspect it.
raw = generate_raw_data()
# display(raw.head())
# display(pd.Series(quality_report(raw), name='count').to_frame())

## 3. Clean the data and re-derive intervention timing

Use the cleaning function and explain why `period`, `post_flag`, `ramp_flag`, and `campaign_flag` should be derived from transaction date rather than trusted blindly from the raw file.

In [ ]:
# TODO: Clean data and confirm date range / row counts.
df = clean_data(raw)
# print(len(df))
# print(df['transaction_date'].min(), df['transaction_date'].max())

## 4. Simple Pre/Post comparison

Calculate Pre and Post KPI summaries. Interpret what changed descriptively, but do not make a causal claim yet.

In [ ]:
# TODO: Create KPI summary and visualize key rates.
# summary = kpi_summary(df)
# display(summary)

## 5. Primary KPI statistical comparison

Run a two-proportion Pre/Post comparison for verification completion. Report Pre rate, Post rate, absolute change in percentage points, relative change, confidence interval, and p-value. Then explain why this still does not prove causality.

In [ ]:
# TODO: Run primary KPI comparison.
# primary = two_proportion_pre_post(df, 'verification_completed')
# display(pd.Series(primary.__dict__).to_frame('value'))

## 6. Full Post vs stable Post

The first 7 days after launch are a ramp period. Compare full Post with stable Post excluding the ramp. Decide whether the launch appears to stabilize after the first week.

In [ ]:
# TODO: Compare full post versus stable post.
# stable = two_proportion_pre_post(df, 'verification_completed', stable_post_mask(df))

## 7. Trend, seasonality, and concurrent events

Aggregate daily. Plot daily verification completion and daily transaction volume. Add launch, ramp, and campaign annotations. Comment on baseline trend, launch shift, post slope, ramp behavior, and campaign effects.

In [ ]:
# TODO: Create daily aggregate and plots.
# daily = daily_aggregate(df)

## 8. Traffic-mix diagnostics

Compare Pre and Post mix for country, device, customer tenure, and risk tier. Identify which mix changes could confound the KPI movement.

In [ ]:
# TODO: Run mix comparison by key dimensions.
# for column in ['country','device_type','customer_tenure','risk_tier']:
#     display(mix_comparison(df, column))

## 9. Secondary metrics and guardrails

Analyze verification time using both mean and median. Then evaluate manual review, payment decline, support contact, and fraud-confirmed rates. Include event counts for rare outcomes such as fraud.

In [ ]:
# TODO: Analyze verification time and guardrails.
# display(pd.Series(continuous_pre_post(df, 'verification_time_seconds')).to_frame('value'))

## 10. Interrupted Time Series

Fit basic and adjusted ITS models for verification completion rate. Interpret the baseline trend, immediate post-launch level change, post-launch slope change, ramp, and campaign terms.

In [ ]:
# TODO: Fit ITS models and interpret key coefficients.
# basic_model, _ = interrupted_time_series(df, adjusted=False)
# adjusted_model, adjusted_daily = interrupted_time_series(df, adjusted=True)

## 11. Segmentation

Analyze completion lift by device, country, customer tenure, and risk tier. Identify whether the observed improvement appears broad-based or concentrated in specific segments. Treat segment findings as diagnostic unless pre-specified.

In [ ]:
# TODO: Segment completion analysis.
# for column in ['device_type','country','customer_tenure','risk_tier']:
#     display(segment_completion(df, column))

## 12. Business impact

Translate the observed/stable effect into annualized business impact. Use a transparent assumption such as annual eligible transaction volume × absolute completion lift. State whether you would use the simple Pre/Post lift, stable Post lift, or adjusted ITS estimate, and why.

In [ ]:
# TODO: Estimate business impact.
# annual_eligible_transactions = 2_000_000
# estimated_incremental_completions = annual_eligible_transactions * <chosen_lift>

## 13. Causal-confidence assessment

Rate the causal confidence as High, Moderate, or Low. Support your rating using the evidence: data quality, baseline trend, ITS results, ramp, campaign, mix shifts, guardrails, and plausible alternative explanations.

## 14. Final recommendation

Write a short stakeholder-ready recommendation using one of these actions: Continue / Scale, Continue with Monitoring, Iterate, Validate Further, or Roll Back. Include:
- What changed
- Why the evidence is or is not credible
- Whether guardrails are acceptable
- Business impact
- Caveats
- Next monitoring steps

## Self-review checklist

Before comparing against the guided notebook or rubric, confirm that your analysis includes:

- [ ] Business question and decision framing
- [ ] Raw data-quality review
- [ ] Clean analytical population with re-derived timing fields
- [ ] Simple Pre/Post KPI comparison
- [ ] Full Post vs stable Post comparison
- [ ] Daily trend and volume visuals
- [ ] Traffic-mix comparison
- [ ] Secondary and guardrail analysis
- [ ] ITS model interpretation
- [ ] Segment analysis
- [ ] Business-impact estimate
- [ ] Causal-confidence rating
- [ ] Clear stakeholder recommendation